# Waste Classification — Colab Training & Evaluation

Trains and compares `efficientnet_b0` vs `convnext_tiny` on a T4 GPU, using the library code from `src/waste_classification/` (Phases 1–2) unchanged — this notebook only orchestrates: load data → train → evaluate → compare → download.

**Before running:** `Runtime > Change runtime type > T4 GPU`.

This notebook is run manually by you, not by Claude Code — see `docs/IMPLEMENTATION_PLAN.md` Phase 3.

## 1. Setup

In [ ]:
import os

REPO_URL = "https://github.com/arghads9177/waste-collection-detection.git"
REPO_DIR = "waste-collection-detection"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}
!git pull

# -e . is required so `import waste_classification` resolves — cloning alone doesn't.
!pip install -e . -q
!pip install -r requirements/requirements-train.txt -q

### Dataset access

Default path: upload the raw `garbage_classification/<class>/*.jpg` folder to Google Drive **once**, then mount Drive here and point `DATA_ROOT` at it — avoids re-uploading ~2GB every session.

Fallback (no Drive upload done yet): use the Kaggle cell further down instead.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Point this at wherever you uploaded garbage_classification/<class>/*.jpg in Drive.
DATA_ROOT = "/content/drive/MyDrive/garbage_classification"

import os
assert os.path.isdir(DATA_ROOT), (
    f"DATA_ROOT not found: {DATA_ROOT} — fix the Drive path, or use the Kaggle "
    "fallback cell below instead."
)

**Fallback — download from Kaggle instead of Drive.** Requires a `kaggle.json` API token. Uncomment and run only if you skipped the Drive upload:

In [ ]:
# from google.colab import files
# files.upload()  # upload kaggle.json when prompted
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d mostafaabla/garbage-classification -p /content/data --unzip
# DATA_ROOT = "/content/data/garbage_classification"

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: no GPU detected — Runtime > Change runtime type > T4 GPU")
else:
    print(torch.cuda.get_device_name(0))

### Split CSVs

These are committed to the repo (`data/processed/splits/`) — pulled straight from the clone above, no regeneration needed. Using the same CSVs as local dev guarantees Colab trains/evaluates on the exact same split (TDD §3 portability: paths are relative to `DATA_ROOT`).

In [ ]:
from pathlib import Path

SPLITS_DIR = Path("data/processed/splits")
assert (SPLITS_DIR / "train.csv").exists(), (
    "Split CSVs missing — did the clone pull data/processed/splits/? "
    "Check .gitignore has the splits exception."
)

with open(SPLITS_DIR / "classes.txt") as f:
    CLASS_NAMES = [line.strip() for line in f if line.strip()]
print(f"{len(CLASS_NAMES)} classes:", CLASS_NAMES)

## 2. Data & Augmentation

In [ ]:
from torch.utils.data import DataLoader

from waste_classification.config import settings
from waste_classification.data.dataset import (
    WasteDataset,
    build_eval_transforms,
    build_train_transforms,
)

IMAGE_SIZE = settings.model_image_size
BATCH_SIZE = settings.train_batch_size

train_dataset = WasteDataset(
    SPLITS_DIR / "train.csv", data_root=DATA_ROOT,
    transform=build_train_transforms(IMAGE_SIZE), class_names=CLASS_NAMES,
)
val_dataset = WasteDataset(
    SPLITS_DIR / "val.csv", data_root=DATA_ROOT,
    transform=build_eval_transforms(IMAGE_SIZE), class_names=CLASS_NAMES,
)
test_dataset = WasteDataset(
    SPLITS_DIR / "test.csv", data_root=DATA_ROOT,
    transform=build_eval_transforms(IMAGE_SIZE), class_names=CLASS_NAMES,
)

print(f"train={len(train_dataset)} val={len(val_dataset)} test={len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# Sanity check: visualize a batch of augmented training images.
import matplotlib.pyplot as plt

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for ax, img, label in zip(axes, images[:6], labels[:6]):
    unnormalized = (img * IMAGENET_STD + IMAGENET_MEAN).clamp(0, 1)
    ax.imshow(unnormalized.permute(1, 2, 0))
    ax.set_title(CLASS_NAMES[label])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Training

Trains both backbones via `build_model()` + `Trainer.fit()` — the exact code proven by the Phase 2 CPU smoke test. Checkpoints every epoch plus `best.pt` (by val accuracy) under `models/checkpoints/<backbone>/`.

In [ ]:
from waste_classification.models.factory import build_model
from waste_classification.training.trainer import Trainer

CHECKPOINT_ROOT = Path("models/checkpoints")
BACKBONES = ["efficientnet_b0", "convnext_tiny"]

trained = {}  # backbone -> {trainer, result}

for backbone in BACKBONES:
    print(f"\n=== Training {backbone} ===")
    model = build_model(backbone, num_classes=len(CLASS_NAMES), pretrained=True)
    trainer = Trainer(
        model=model,
        backbone=backbone,
        class_names=CLASS_NAMES,
        train_loader=train_loader,
        val_loader=val_loader,
        checkpoint_dir=CHECKPOINT_ROOT / backbone,
        lr=settings.train_lr,
        early_stopping_patience=settings.train_early_stopping_patience,
        device=DEVICE,
    )
    result = trainer.fit(
        warmup_epochs=settings.train_warmup_epochs,
        fine_tune_epochs=max(settings.train_epochs - settings.train_warmup_epochs, 0),
    )
    trained[backbone] = {"trainer": trainer, "result": result}
    print(f"{backbone} best val accuracy: {result['best_val_accuracy']:.4f}")

In [ ]:
# Persist per-epoch training history (feeds outputs/training_runs/ in the zip at the end).
import pandas as pd

OUTPUT_DIR = Path("outputs")
(OUTPUT_DIR / "training_runs").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "eval_reports").mkdir(parents=True, exist_ok=True)

for backbone, run in trained.items():
    history_df = pd.DataFrame(run["result"]["history"])
    history_df.to_csv(OUTPUT_DIR / "training_runs" / f"{backbone}_history.csv", index=False)
    display(history_df)

## 4. Evaluation

Runs `compute_classification_metrics()` per backbone on the **test** split (held out from both training and early-stopping decisions), using each backbone's `best.pt`.

In [ ]:
import json

from waste_classification.models.factory import load_checkpoint
from waste_classification.training.metrics import compute_classification_metrics


def evaluate_on_test(checkpoint_path, loader, device):
    model, checkpoint = load_checkpoint(checkpoint_path, device=device)
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            all_preds.extend(outputs.argmax(dim=1).cpu().tolist())
            all_labels.extend(labels.tolist())
    metrics = compute_classification_metrics(all_labels, all_preds, checkpoint["class_names"])
    return metrics


test_results = {}
for backbone in BACKBONES:
    best_ckpt = CHECKPOINT_ROOT / backbone / "best.pt"
    metrics = evaluate_on_test(best_ckpt, test_loader, DEVICE)
    test_results[backbone] = metrics
    with open(OUTPUT_DIR / "eval_reports" / f"{backbone}_test_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    print(f"\n=== {backbone} test metrics ===")
    print(f"accuracy={metrics['accuracy']:.4f} macro_f1={metrics['macro_f1']:.4f}")

In [ ]:
# Inline per-class metric tables + confusion matrices.
import numpy as np

for backbone, metrics in test_results.items():
    print(f"\n{backbone} — per-class metrics:")
    display(pd.DataFrame(metrics["per_class"]).T)

    cm = pd.DataFrame(metrics["confusion_matrix"], index=CLASS_NAMES, columns=CLASS_NAMES)
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(CLASS_NAMES)))
    ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels(CLASS_NAMES, rotation=90)
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{backbone} confusion matrix (test)")
    fig.colorbar(im)
    plt.tight_layout()
    plt.show()

## 5. Per-Class F1 Check — Iterate, Don't One-Shot

Verify per-class F1 ≥ 0.75 for **all 12 classes**. Given dataset imbalance, expect at least one class to miss on the first pass — that's normal, not a bug.

In [ ]:
F1_TARGET = 0.75


def failing_classes(metrics, target=F1_TARGET):
    return {cls: v["f1"] for cls, v in metrics["per_class"].items() if v["f1"] < target}


for backbone, metrics in test_results.items():
    failing = failing_classes(metrics)
    if failing:
        print(f"{backbone}: {len(failing)} class(es) below F1 {F1_TARGET}: {failing}")
    else:
        print(f"{backbone}: all {len(CLASS_NAMES)} classes meet F1 >= {F1_TARGET}")

### If any class misses the bar: remediation ladder

Work down this ladder — re-running **§3 Training → §4 Evaluation → §5 this check** after each change — before declaring a winner in §6:

1. **Class-weighted loss** — compute inverse-frequency weights from `train_dataset` (helper cell below) and pass `class_weights=` to `Trainer(...)`.
2. **More fine-tune epochs / lower LR** — raise `fine_tune_epochs` or halve `settings.train_lr` for the failing backbone and re-run §3 for that backbone only.
3. **Targeted augmentation** — strengthen `build_train_transforms()` for the weak classes (e.g. wider rotation/crop scale) in `src/waste_classification/data/dataset.py`.

If you exhaust the ladder and a class still misses, document the shortfall and why in §6 rather than silently accepting it.

In [ ]:
# Helper: inverse-frequency class weights, ready to pass into Trainer(class_weights=...).
from collections import Counter


def compute_class_weights(dataset, class_names):
    counts = Counter(dataset.df["label"])
    total = sum(counts.values())
    weights = [total / (len(class_names) * counts.get(cls, 1)) for cls in class_names]
    return torch.tensor(weights, dtype=torch.float32)


# Example (uncomment if remediation step 1 is needed):
# class_weights = compute_class_weights(train_dataset, CLASS_NAMES)
# trainer = Trainer(..., class_weights=class_weights)

## 6. Model Comparison & Selection

Side-by-side accuracy, macro-F1, param count, and a **rough** GPU latency estimate (the authoritative CPU latency benchmark happens later in Phase 4, against the exported ONNX model — this number is only for picking a backbone now).

In [ ]:
import time


def estimate_latency_ms(model, device, image_size, n=20):
    model.eval()
    dummy = torch.randn(1, 3, image_size, image_size, device=device)
    with torch.no_grad():
        for _ in range(5):
            model(dummy)
        if device == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(n):
            model(dummy)
        if device == "cuda":
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
    return (elapsed / n) * 1000


comparison_rows = []
for backbone in BACKBONES:
    model, checkpoint = load_checkpoint(CHECKPOINT_ROOT / backbone / "best.pt", device=DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    latency_ms = estimate_latency_ms(model, DEVICE, IMAGE_SIZE)
    comparison_rows.append({
        "backbone": backbone,
        "accuracy": test_results[backbone]["accuracy"],
        "macro_f1": test_results[backbone]["macro_f1"],
        "params_millions": n_params / 1e6,
        "rough_gpu_latency_ms": latency_ms,
    })

comparison_df = pd.DataFrame(comparison_rows).set_index("backbone")
display(comparison_df)

### Winning backbone

_Fill in after reviewing the table above:_

**Winner:** `<efficientnet_b0 | convnext_tiny>`

**Why:** _...macro-F1 / per-class F1 bar / latency trade-off..._

## 7. Download Artifacts

Zips `models/checkpoints/`, `outputs/training_runs/`, `outputs/eval_reports/` preserving their repo-relative paths, so you can unzip directly at the repo root locally. Phase 4 picks up from `models/checkpoints/<winning_backbone>/best.pt`.

In [ ]:
import shutil

from google.colab import files

ARTIFACT_DIRS = ["models/checkpoints", "outputs/training_runs", "outputs/eval_reports"]
STAGE_DIR = Path("/content/artifact_staging")
if STAGE_DIR.exists():
    shutil.rmtree(STAGE_DIR)

for d in ARTIFACT_DIRS:
    dest = STAGE_DIR / d
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copytree(d, dest, dirs_exist_ok=True)

archive_path = shutil.make_archive("/content/waste_classification_artifacts", "zip", root_dir=str(STAGE_DIR))
print("Archive ready:", archive_path)
files.download(archive_path)